In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal
import os
from typing import TypedDict, Annotated
import operator
from operator import add
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

if not os.getenv('GOOGLE_API_KEY'):
    raise ValueError('Add GOOGLE_API_KEY to a .env file before running.')

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
);

In [ ]:
class JokeState(TypedDict):
    topic : str
    joke: Annotated[list[str], add]
    explain: Annotated[list[str], add]


In [ ]:
def generate_joke(state: JokeState):
    topic = state["topic"]
    prompt = f"Write a short, funny joke about {topic}"
    response = llm.invoke(prompt)
    return {"joke": [response.content]}

In [ ]:
def generate_explanation(state: JokeState):
    joke = state["joke"][-1]
    prompt = f"Explain this joke simply in 1-2 sentences: {joke}"
    response = llm.invoke(prompt)
    return {"explain": [response.content]}

In [ ]:

Graph = StateGraph(JokeState)

Graph.add_node("generate_joke", generate_joke)
Graph.add_node("generate_explanation", generate_explanation)

Graph.add_edge(START, "generate_joke")
Graph.add_edge("generate_joke", "generate_explanation")
Graph.add_edge("generate_explanation", END)

In [ ]:

checkpointer = InMemorySaver()
bot = Graph.compile(checkpointer=checkpointer)

config1 = {"configurable": {"thread_id": "1"}}

res1 = bot.invoke({"topic": ["football"]}, config=config1)

print("Thread 1 - weather")
print("Joke:", res1["joke"][-1])
print("Explanation:", res1["explain"][-1])

config2 = {"configurable": {"thread_id": "2"}}

res2 = bot.invoke({"topic": ["government"]}, config=config2)

print("\nThread 2 - Python")
print("Joke:", res2["joke"][-1])
print("Explanation:", res2["explain"][-1])

res3 = bot.invoke({"topic": ["university"]}, config=config1)

print("\nThread 1 - After Second Run")
print("All Topics:", res3["topic"])
print("All Jokes:", res3["joke"])
print("All Explanations:", res3["explain"])

Thread 1 - Football
Joke: [{'type': 'text', 'text': 'Why are football players so good at staying cool?\n\nBecause they have a whole team of fans behind them!', 'extras': {'signature': 'EmAKXgFpFH0TkAdNopBaETry8DgQQQMCSDDEkbhrT6NqkU+QjVl0VPh3SgAPiGotN9ZkhQwuD8XhVlxUO6EGZJQdJRD1n0mOHX8tBhLCDW4aWhqTuB5rDHA9QvxgtwKup1g='}}]
Explanation: [{'type': 'text', 'text': 'The joke plays on the double meaning of the word "fans"—athletic supporters in the stands and electric cooling devices. Football players stay cool because they have both a crowd cheering for them and actual breeze-blowing fans behind them.', 'extras': {'signature': 'EmAKXgFpFH0TB1gr5aevZRo4W6Q4+OMwhygABevpfZM4bR3QeKyVyBpg9JH6GvpGAmBs+Nv6ahWRtdsFtZM8LvKeBWpF3m3CSLEBn47JvvZ+eFgnICuz00JvkMuwWE2hbqI='}}]

Thread 2 - Python
Joke: [{'type': 'text', 'text': "Why do Python programmers wear glasses? \n\nBecause they don't C#!", 'extras': {'signature': 'EmAKXgFpFH0ToTZE/gMQKSV6a9qOd2h4TjJTBBkBWrZbHIkyivw04WxSmcHtxgoFNuxqeV8XYakqQLTYb0+ed4GK

In [5]:
snapshot1 = bot.get_state(config1)
snapshot2 = bot.get_state(config2)

print("Thread 1 Saved State:", snapshot1.values)
print("Thread 2 Saved State:", snapshot2.values)

Thread 1 Saved State: {'topic': ['football', 'cricket'], 'joke': [[{'type': 'text', 'text': 'Why are football players so good at staying cool?\n\nBecause they have a whole team of fans behind them!', 'extras': {'signature': 'EmAKXgFpFH0TkAdNopBaETry8DgQQQMCSDDEkbhrT6NqkU+QjVl0VPh3SgAPiGotN9ZkhQwuD8XhVlxUO6EGZJQdJRD1n0mOHX8tBhLCDW4aWhqTuB5rDHA9QvxgtwKup1g='}}], [{'type': 'text', 'text': "Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they know they'll be dismissed for a duck!", 'extras': {'signature': 'EmAKXgFpFH0TaqpJAjlIK//bRcHj55yhygjxy/Jbffa5EDlH1/K73cP3xIzKw7jDV3lqA9lhO4aHzEOJjIAdC32TA7NdB/bPkjJ/yqATN+f2c4bJ1sl4zaprheZXF+xDUNY='}}]], 'explain': [[{'type': 'text', 'text': 'The joke plays on the double meaning of the word "fans"—athletic supporters in the stands and electric cooling devices. Football players stay cool because they have both a crowd cheering for them and actual breeze-blowing fans behind them.', 'extras': {'signature': 'EmAK